In [13]:
import pandas as pd
import time
import logging
from pathlib import Path
from sqlalchemy import create_engine, text


In [14]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    BASE_DIR = current_path.parent
else:
    BASE_DIR = current_path

DATA_DIR = BASE_DIR / "data"
DATABASE_DIR = BASE_DIR / "database"

DATABASE_DIR.mkdir(exist_ok=True)

DATABASE_PATH = DATABASE_DIR / "vendor_iq.db"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Database path:", DATABASE_PATH)

Base directory: d:\VendorIQ
Data directory: d:\VendorIQ\data
Database path: d:\VendorIQ\database\vendor_iq.db


In [15]:
engine = create_engine(f"sqlite:///{DATABASE_PATH.as_posix()}", echo=False)

In [16]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [17]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

for file in csv_files:
    print(file.name)

begin_inventory.csv
end_inventory.csv
purchase_prices.csv
purchases.csv
sales.csv
vendor_invoice.csv


In [18]:
def ingest_csv_to_db(file_path, engine, chunksize=100000):
    """
    Reads a CSV file in chunks and ingests it into the SQLite database.
    The table name is created from the CSV file name.
    """

    table_name = file_path.stem
    total_rows = 0
    first_chunk = True

    logging.info(f"Starting ingestion for: {file_path.name}")

    for chunk in pd.read_csv(file_path, chunksize=chunksize, low_memory=False):
        if first_chunk:
            if_exists_mode = "replace"
            first_chunk = False
        else:
            if_exists_mode = "append"

        chunk.to_sql(
            table_name,
            con=engine,
            if_exists=if_exists_mode,
            index=False
        )

        total_rows += len(chunk)
        logging.info(f"{table_name}: {total_rows:,} rows loaded")

    logging.info(f"Completed ingestion for {table_name}. Total rows: {total_rows:,}")

In [19]:
def load_raw_data():
    """
    Loads all CSV files from the data folder into the SQLite database.
    """

    start_time = time.time()

    for file_path in sorted(DATA_DIR.glob("*.csv")):
        ingest_csv_to_db(file_path, engine)

    end_time = time.time()
    total_minutes = round((end_time - start_time) / 60, 2)

    logging.info("------------- Ingestion Complete -------------")
    logging.info(f"Total Time Taken: {total_minutes} minutes")

In [20]:
load_raw_data()

2026-07-02 12:39:39,926 - INFO - Starting ingestion for: begin_inventory.csv
2026-07-02 12:39:42,306 - INFO - begin_inventory: 100,000 rows loaded
2026-07-02 12:39:44,435 - INFO - begin_inventory: 200,000 rows loaded
2026-07-02 12:39:44,617 - INFO - begin_inventory: 206,529 rows loaded
2026-07-02 12:39:44,620 - INFO - Completed ingestion for begin_inventory. Total rows: 206,529
2026-07-02 12:39:44,625 - INFO - Starting ingestion for: end_inventory.csv
2026-07-02 12:39:46,760 - INFO - end_inventory: 100,000 rows loaded
2026-07-02 12:39:48,901 - INFO - end_inventory: 200,000 rows loaded
2026-07-02 12:39:49,542 - INFO - end_inventory: 224,489 rows loaded
2026-07-02 12:39:49,546 - INFO - Completed ingestion for end_inventory. Total rows: 224,489
2026-07-02 12:39:49,549 - INFO - Starting ingestion for: purchase_prices.csv
2026-07-02 12:39:49,819 - INFO - purchase_prices: 12,261 rows loaded
2026-07-02 12:39:49,822 - INFO - Completed ingestion for purchase_prices. Total rows: 12,261
2026-07-0